In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv('dataset.csv')

In [3]:
DROP_COLS = [
    'comment', 'map_file', 'meta_data', 'direction', 'route_path',
    'citizen_accident_id', 'assigned_to_police_id',
    'age_of_truck', 'reason_breakdown', 'cargo_material',
    'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude',
    'resolved_by_id', 'resolved_datetime',
    'description', 'kgid', 'veh_no',
    'id', 'address', 'end_address',  # free text / ID
    'map_file', 'last_modified_by_id', 'created_by_id',
    'modified_datetime', 'created_date',  # leakage risk / not features
    'closed_by_id', 'police_station', 'client_id', 'authenticated'
]
df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

In [4]:
df_clean['closed_datetime']   = pd.to_datetime(df_clean['closed_datetime'],   utc=True, errors='coerce')
df_clean['start_datetime']    = pd.to_datetime(df_clean['start_datetime'],     utc=True, errors='coerce')
df_clean['resolved_datetime_fallback'] = pd.to_datetime(
    df.get('resolved_datetime', pd.Series()), utc=True, errors='coerce'
)

# Use closed_datetime first, fall back to resolved_datetime
df_clean['end_ts'] = df_clean['closed_datetime'].fillna(df_clean['resolved_datetime_fallback'])
df_clean['duration_mins'] = (
    df_clean['end_ts'] - df_clean['start_datetime']
).dt.total_seconds() / 60

In [5]:
df_clean['severity'] = pd.cut(
    df_clean['duration_mins'],
    bins=[0, 30, 90, 240, 1440],
    labels=['low', 'medium', 'high', 'critical']
)

In [6]:
df_clean['hour']        = df_clean['start_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['start_datetime'].dt.dayofweek
df_clean['month']       = df_clean['start_datetime'].dt.month
df_clean['is_weekend']  = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['is_peak']     = df_clean['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df_clean['is_night']    = df_clean['hour'].isin(range(22, 24)).astype(int)

# ── 5. Geo features ───────────────────────────────────────────────────────────
df_clean['lat_bin'] = pd.cut(df_clean['latitude'],  bins=8, labels=False)
df_clean['lon_bin'] = pd.cut(df_clean['longitude'], bins=8, labels=False)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0) != 0).astype(int)


In [7]:
CAT_FILL_UNKNOWN = ['zone', 'junction', 'gba_identifier', 'veh_type', 'corridor']
for col in CAT_FILL_UNKNOWN:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['priority'] = df_clean['priority'].fillna(df_clean['priority'].mode()[0])
df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

In [9]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [10]:
LE_COLS = ['event_type', 'event_cause', 'veh_type', 'status']
le_encoders = {}
for col in LE_COLS:
    if col in df_clean.columns:
        le = LabelEncoder()
        df_clean[col + '_enc'] = le.fit_transform(df_clean[col].astype(str))
        le_encoders[col] = le

# High-cardinality: target encoding for zone, junction
# (avoids OHE explosion; robust for tree models)
target_col = 'duration_mins'
for col in ['zone', 'junction', 'corridor', 'gba_identifier']:
    if col in df_clean.columns:
        mean_map = df_clean.groupby(col)[target_col].mean()
        df_clean[col + '_tenc'] = df_clean[col].map(mean_map)
        # Add binary "is_known" flag alongside
        df_clean[col + '_known'] = (df_clean[col] != 'Unknown').astype(int)

# Priority ordinal encoding
priority_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
if 'priority' in df_clean.columns:
    df_clean['priority_enc'] = df_clean['priority'].str.lower().map(priority_map)

In [11]:
FEATURE_COLS = [
    # temporal
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak', 'is_night',
    # geo
    'latitude', 'longitude', 'lat_bin', 'lon_bin', 'has_end_location',
    # event attributes
    'event_type_enc', 'event_cause_enc', 'requires_road_closure',
    'veh_type_enc', 'status_enc',
    # encoded high-card
    'zone_tenc', 'zone_known',
    'junction_tenc', 'junction_known',
    'corridor_tenc', 'corridor_known',
    'gba_identifier_tenc', 'gba_identifier_known',
    # priority (when not the target)
    'priority_enc',
]
# Keep only columns that exist after all the steps above
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_clean.columns]

In [12]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)
import xgboost as xgb
import lightgbm as lgb

In [16]:
# ── Rebuild severity with explicit NaN drop ───────────────────────────────────
df_clean['severity'] = pd.cut(
    df_clean['duration_mins'],
    bins=[0, 30, 90, 240, 1440],
    labels=['low', 'medium', 'high', 'critical'],
    include_lowest=True   # <-- this catches the exact 0 boundary
)

# Drop any rows where severity is still NaN (edge cases above 1440 or below 0)
df_clean = df_clean[df_clean['severity'].notna()].copy()

# Encode as integer starting from 0 — NO .cat.codes (it gives -1 for NaN)
severity_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
df_clean['severity_enc'] = df_clean['severity'].map(severity_map)

# Verify
print("Severity distribution:")
print(df_clean['severity_enc'].value_counts().sort_index())
print(f"Any NaN? {df_clean['severity_enc'].isna().sum()}")
print(f"Unique classes: {sorted(df_clean['severity_enc'].unique())}")
# Must show: [0, 1, 2, 3] — nothing else

Severity distribution:
severity_enc
0     848
1    1085
2     432
3     158
Name: count, dtype: int64
Any NaN? 0
Unique classes: [0, 1, 2, 3]


In [17]:
X = df_clean[FEATURE_COLS].fillna(-999)   # -999 sentinel for trees
y_reg   = df_clean['duration_mins']
y_clf_b = df_clean['requires_road_closure']   # binary
y_clf_m = df_clean['severity'].cat.codes       # multi-class (0=low,1=med,2=high,3=critical)

X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_b_train, y_b_test = train_test_split(X, y_clf_b, test_size=0.2, random_state=42)
_, _, y_m_train, y_m_test = train_test_split(X, y_clf_m, test_size=0.2, random_state=42)

def regression_report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE : {mae:.2f} mins")
    print(f"  RMSE: {rmse:.2f} mins")
    print(f"  R²  : {r2:.4f}")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

def classification_report_short(name, y_true, y_pred, y_prob=None):
    f1 = f1_score(y_true, y_pred, average='weighted')
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted') \
          if y_prob is not None and y_prob.ndim == 2 else None
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  F1 (weighted): {f1:.4f}")
    if auc: print(f"  ROC-AUC:       {auc:.4f}")
    return {'model': name, 'F1': f1, 'AUC': auc}

KeyError: "['hour', 'day_of_week', 'month', 'is_weekend', 'is_peak', 'is_night', 'lat_bin', 'lon_bin', 'has_end_location', 'event_type_enc', 'event_cause_enc', 'veh_type_enc', 'status_enc', 'zone_tenc', 'zone_known', 'junction_tenc', 'junction_known', 'corridor_tenc', 'corridor_known', 'gba_identifier_tenc', 'gba_identifier_known', 'priority_enc'] not in index"

Usable rows after duration filter: 2523
NaN in duration_mins: 0


In [4]:
print("\n" + "="*50)
print("  TASK C: PRIORITY (MULTI-CLASS)")
print("="*50)

# Remove priority_enc from features when it's the target (avoid leakage!)
feat_no_priority = [c for c in FEATURE_COLS if c != 'priority_enc']
X_tr_m = X_train[feat_no_priority]
X_te_m = X_test[feat_no_priority]

print(f"\nClass distribution:\n{pd.Series(y_m_train).value_counts()}")

results_m = []

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_mc = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_mc.fit(X_tr_m, y_m_train)
pred  = rf_mc.predict(X_te_m)
prob  = rf_mc.predict_proba(X_te_m)
f1    = f1_score(y_m_test, pred, average='weighted')
auc   = roc_auc_score(y_m_test, prob, multi_class='ovr', average='weighted')
results_m.append({'model': 'RandomForest', 'F1': f1, 'AUC': auc})
print(f"\nRF  F1={f1:.4f}  AUC={auc:.4f}")

# ── XGBoost multi-class ───────────────────────────────────────────────────────
xgb_mc = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_m_train)),
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric='mlogloss'
)
xgb_mc.fit(X_tr_m, y_m_train, eval_set=[(X_te_m, y_m_test)], verbose=False)
pred  = xgb_mc.predict(X_te_m)
prob  = xgb_mc.predict_proba(X_te_m)
f1    = f1_score(y_m_test, pred, average='weighted')
auc   = roc_auc_score(y_m_test, prob, multi_class='ovr', average='weighted')
results_m.append({'model': 'XGBoost', 'F1': f1, 'AUC': auc})
print(f"XGB F1={f1:.4f}  AUC={auc:.4f}")

# ── LightGBM multi-class ──────────────────────────────────────────────────────
lgb_mc = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(np.unique(y_m_train)),
    n_estimators=500, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_mc.fit(
    X_tr_m, y_m_train,
    eval_set=[(X_te_m, y_m_test)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)
pred  = lgb_mc.predict(X_te_m)
prob  = lgb_mc.predict_proba(X_te_m)
f1    = f1_score(y_m_test, pred, average='weighted')
auc   = roc_auc_score(y_m_test, prob, multi_class='ovr', average='weighted')
results_m.append({'model': 'LightGBM', 'F1': f1, 'AUC': auc})
print(f"LGB F1={f1:.4f}  AUC={auc:.4f}")

print("\nTask C Summary:")
print(pd.DataFrame(results_m).sort_values('AUC', ascending=False).to_string(index=False))


  TASK C: PRIORITY (MULTI-CLASS)


NameError: name 'X_train' is not defined

In [5]:
# file3_quantile_huber_loss.py
# Technique: optimize MAE directly using quantile regression (q=0.5 = median = MAE)
# + Huber loss as comparison. Both are more robust to long-tail outliers than MSE.

import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─── [Paste your full preprocessing from file1 here, up to X / y_raw] ────────
# For brevity: assumes df_clean, FEATURE_COLS, X, y_raw are ready

# ─── Rebuild if needed ────────────────────────────────────────────────────────
for col in ['closed_datetime','start_datetime','resolved_datetime','modified_datetime','created_date']:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')
df['end_ts']        = df['closed_datetime'].fillna(df['resolved_datetime'])
df['duration_mins'] = (df['end_ts'] - df['start_datetime']).dt.total_seconds() / 60
df['severity']     = pd.cut(df['duration_mins'], bins=[0,30,90,240,1440],
                             labels=['low','medium','high','critical'], include_lowest=True)
df['severity_enc'] = df['severity'].map({'low':0,'medium':1,'high':2,'critical':3})
mask = (df['duration_mins'].notna() & df['severity_enc'].notna() &
        (df['duration_mins']>0) & (df['duration_mins']<1440) &
        df['priority'].notna() & df['address'].notna())
df_clean = df[mask].copy()

for col in ['zone','junction','gba_identifier','veh_type','corridor','event_cause','event_type','status']:
    df_clean[col] = df_clean[col].fillna('Unknown')
df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

df_clean['hour']        = df_clean['start_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['start_datetime'].dt.dayofweek
df_clean['month']       = df_clean['start_datetime'].dt.month
df_clean['is_weekend']  = df_clean['day_of_week'].isin([5,6]).astype(int)
df_clean['is_peak']     = df_clean['hour'].isin([7,8,9,17,18,19]).astype(int)
df_clean['is_night']    = df_clean['hour'].isin([22,23,0,1,2]).astype(int)
df_clean['hour_sin']    = np.sin(2*np.pi*df_clean['hour']/24)
df_clean['hour_cos']    = np.cos(2*np.pi*df_clean['hour']/24)
df_clean['dow_sin']     = np.sin(2*np.pi*df_clean['day_of_week']/7)
df_clean['dow_cos']     = np.cos(2*np.pi*df_clean['day_of_week']/7)
df_clean['dist_from_center'] = np.sqrt(
    (df_clean['latitude']-12.9716)**2+(df_clean['longitude']-77.5946)**2)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0)!=0).astype(int)
df_clean['lat_bin'] = pd.cut(df_clean['latitude'], bins=10, labels=False)
df_clean['lon_bin'] = pd.cut(df_clean['longitude'], bins=10, labels=False)
df_clean['geo_span'] = np.sqrt(
    (df_clean['endlatitude'].fillna(df_clean['latitude'])-df_clean['latitude'])**2+
    (df_clean['endlongitude'].fillna(df_clean['longitude'])-df_clean['longitude'])**2)

LE_COLS = ['event_type','event_cause','veh_type','status']
le_encoders = {}
for col in LE_COLS:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    le_encoders[col] = le
df_clean['priority_enc'] = df_clean['priority'].str.lower().map(
    {'low':0,'medium':1,'high':2,'critical':3}).fillna(1).astype(int)

for col in ['zone','junction','corridor','gba_identifier','police_station']:
    if col in df_clean.columns:
        mean_map = df_clean.groupby(col)['duration_mins'].mean()
        df_clean[f'{col}_tenc'] = df_clean[col].map(mean_map).fillna(df_clean['duration_mins'].mean())
        df_clean[f'{col}_known'] = (df_clean[col] != 'Unknown').astype(int)

# Interaction features
df_clean['cause_x_peak']    = df_clean['event_cause_enc'] * df_clean['is_peak']
df_clean['cause_x_closure'] = df_clean['event_cause_enc'] * df_clean['requires_road_closure']
df_clean['zone_x_peak']     = df_clean['zone_tenc'] * df_clean['is_peak']
df_clean['cause_hour_mean'] = df_clean.groupby(
    ['event_cause','hour'])['duration_mins'].transform('mean')

FEATURE_COLS = [
    'hour','day_of_week','month','is_weekend','is_peak','is_night',
    'hour_sin','hour_cos','dow_sin','dow_cos',
    'latitude','longitude','lat_bin','lon_bin','has_end_location','geo_span','dist_from_center',
    'event_type_enc','event_cause_enc','requires_road_closure','veh_type_enc','status_enc','priority_enc',
    'zone_tenc','zone_known','junction_tenc','junction_known',
    'corridor_tenc','corridor_known','gba_identifier_tenc','gba_identifier_known',
    'cause_x_peak','cause_x_closure','zone_x_peak','cause_hour_mean',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_clean.columns]
X     = df_clean[FEATURE_COLS].fillna(-999)
y_raw = df_clean['duration_mins']
y_log = np.log1p(y_raw)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = {}

# ──────────────────────────────────────────────────────────────────────────────
#  METHOD A: Quantile regression at q=0.5 (direct MAE minimization)
#  This is mathematically equivalent to minimizing MAE
# ──────────────────────────────────────────────────────────────────────────────
print("\nMethod A: Quantile regression (q=0.5)")

def objective_quantile(trial):
    params = {
        'objective':         'reg:quantileerror',
        'quantile_alpha':    0.5,            # median = MAE minimization
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1500),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state': 42, 'verbosity': 0, 'n_jobs': -1,
    }
    maes = []
    for tr_idx, va_idx in kf.split(X):
        m = xgb.XGBRegressor(**params)
        m.fit(X.iloc[tr_idx], y_raw.iloc[tr_idx],
              eval_set=[(X.iloc[va_idx], y_raw.iloc[va_idx])],
              verbose=False)
        maes.append(mean_absolute_error(y_raw.iloc[va_idx], m.predict(X.iloc[va_idx])))
    return np.mean(maes)

study_q = optuna.create_study(direction='minimize',
                               sampler=optuna.samplers.TPESampler(seed=42))
study_q.optimize(objective_quantile, n_trials=80, show_progress_bar=True)
best_q = study_q.best_params
print(f"Quantile best OOF MAE: {study_q.best_value:.2f}")

oof_q = np.zeros(len(X))
for tr_idx, va_idx in kf.split(X):
    m = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.5,
                          **best_q, verbosity=0, n_jobs=-1)
    m.fit(X.iloc[tr_idx], y_raw.iloc[tr_idx])
    oof_q[va_idx] = m.predict(X.iloc[va_idx])
results['Quantile(q=0.5)'] = {
    'MAE': mean_absolute_error(y_raw, oof_q),
    'RMSE': np.sqrt(mean_squared_error(y_raw, oof_q)),
    'R2': r2_score(y_raw, oof_q)
}

# ──────────────────────────────────────────────────────────────────────────────
#  METHOD B: Pseudo-Huber loss (smooth MAE — less outlier sensitive than MSE)
# ──────────────────────────────────────────────────────────────────────────────
print("\nMethod B: Pseudo-Huber loss")

def objective_huber(trial):
    params = {
        'objective':    'reg:pseudohubererror',
        'huber_slope':  trial.suggest_float('huber_slope', 0.5, 20.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth':    trial.suggest_int('max_depth', 4, 10),
        'learning_rate':trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'subsample':    trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':    trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda':   trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state': 42, 'verbosity': 0, 'n_jobs': -1,
    }
    maes = []
    for tr_idx, va_idx in kf.split(X):
        m = xgb.XGBRegressor(**params)
        m.fit(X.iloc[tr_idx], y_raw.iloc[tr_idx],
              eval_set=[(X.iloc[va_idx], y_raw.iloc[va_idx])],
              verbose=False)
        maes.append(mean_absolute_error(y_raw.iloc[va_idx], m.predict(X.iloc[va_idx])))
    return np.mean(maes)

study_h = optuna.create_study(direction='minimize',
                               sampler=optuna.samplers.TPESampler(seed=42))
study_h.optimize(objective_huber, n_trials=80, show_progress_bar=True)
best_h = study_h.best_params
print(f"Huber best OOF MAE: {study_h.best_value:.2f}")

oof_h = np.zeros(len(X))
for tr_idx, va_idx in kf.split(X):
    m = xgb.XGBRegressor(objective='reg:pseudohubererror', **best_h, verbosity=0, n_jobs=-1)
    m.fit(X.iloc[tr_idx], y_raw.iloc[tr_idx])
    oof_h[va_idx] = m.predict(X.iloc[va_idx])
results['Huber'] = {
    'MAE': mean_absolute_error(y_raw, oof_h),
    'RMSE': np.sqrt(mean_squared_error(y_raw, oof_h)),
    'R2': r2_score(y_raw, oof_h)
}

# ──────────────────────────────────────────────────────────────────────────────
#  METHOD C: Log-transform + quantile loss (best of both worlds)
# ──────────────────────────────────────────────────────────────────────────────
print("\nMethod C: Log-transform + Quantile regression")

oof_lq = np.zeros(len(X))
for tr_idx, va_idx in kf.split(X):
    m = xgb.XGBRegressor(
        objective='reg:quantileerror', quantile_alpha=0.5,
        **{k: v for k, v in best_q.items() if k not in ['objective','quantile_alpha']},
        n_estimators=600, learning_rate=0.05,
        verbosity=0, n_jobs=-1, random_state=42
    )
    m.fit(X.iloc[tr_idx], y_log.iloc[tr_idx])
    oof_lq[va_idx] = np.expm1(m.predict(X.iloc[va_idx]))
results['Log+Quantile'] = {
    'MAE': mean_absolute_error(y_raw, oof_lq),
    'RMSE': np.sqrt(mean_squared_error(y_raw, oof_lq)),
    'R2': r2_score(y_raw, oof_lq)
}

# ─── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"  FILE 3 RESULTS — Loss function comparison")
print(f"{'='*50}")
for name, r in sorted(results.items(), key=lambda x: x[1]['MAE']):
    print(f"  {name:20s}  MAE={r['MAE']:.2f}  RMSE={r['RMSE']:.2f}  R²={r['R2']:.4f}")

best_method = min(results, key=lambda x: results[x]['MAE'])
print(f"\n  Winner: {best_method}")

os.makedirs('models_v2', exist_ok=True)
best_model_f3 = xgb.XGBRegressor(
    objective='reg:quantileerror', quantile_alpha=0.5,
    **best_q, verbosity=0, n_jobs=-1
)
best_model_f3.fit(X, y_raw)
joblib.dump(best_model_f3, 'models_v2/xgb_quantile.pkl')
print("Saved → models_v2/xgb_quantile.pkl")


Method A: Quantile regression (q=0.5)


  0%|          | 0/80 [00:00<?, ?it/s]

Quantile best OOF MAE: 66.52

Method B: Pseudo-Huber loss


  0%|          | 0/80 [00:00<?, ?it/s]

Huber best OOF MAE: 69.43

Method C: Log-transform + Quantile regression


TypeError: xgboost.sklearn.XGBRegressor() got multiple values for keyword argument 'n_estimators'